# `agmini` from a notebook

Two functions:

- `collect(root, include=, exclude=)` — the list of files that would be sent
- `pack(root, out, include=, exclude=)` — writes the QR sheets, returns a summary

Decode on the other machine with `agunpack.py`; `demo.ipynb` walks that half.

In [1]:
import agmini
help(agmini.pack)

Help on function pack in module agmini:

pack(root, out='sheets.html', include=None, exclude=(), chunk=600, cols=4, rows=2, scale=4, max_size=1048576, secrets=False)
    Write QR sheets for `root` to `out`. Returns a summary dict.
    
    chunk    bytes of payload per QR symbol
    cols,rows  symbols per sheet
    scale    screen pixels per QR module - the setting that decides whether a
             photo can be decoded at all. Below ~3 the display cannot show the
             detail. Defaults suit a 1080p screen; on a 1440p+ display try
             chunk=1100, or cols=6, rows=3 on 4K.



## A tree to try it on

Real modules from your own standard library, so the compression ratio is honest —
synthetic filler compresses much better than actual source and would flatter the
estimate.

In [2]:
import pathlib, shutil, sysconfig, tempfile

ROOT = pathlib.Path(tempfile.mkdtemp(prefix="agmini-")) / "repo"
(ROOT / "src").mkdir(parents=True)
(ROOT / "tests").mkdir()

stdlib = pathlib.Path(sysconfig.get_paths()["stdlib"])
n = 0
for p in sorted(stdlib.glob("*.py")):
    if 3_000 < p.stat().st_size < 40_000 and n < 12:
        shutil.copy(p, ROOT / "src" / p.name); n += 1

(ROOT / "README.md").write_text("# demo\n")
(ROOT / "tests" / "test_app.py").write_text("def test(): assert True\n")
(ROOT / ".env").write_text("TOKEN=sk-secret\n")
print(ROOT)

/tmp/agmini-3tjjhjcg/repo


## 1. `collect` — check the filters before committing

`include` and `exclude` take extensions (`".py"`) and globs (`"*_test.py"`) in the
same list. `include=None` means every text file.

In [3]:
print("everything      :", len(agmini.collect(ROOT)))
print("only .py        :", len(agmini.collect(ROOT, include=[".py"])))
print("py+md, no tests :", agmini.collect(ROOT, include=[".py", ".md"],
                                          exclude=["test_*.py"])[:4], "...")

# .env is skipped before `include` is applied, so no filter can let it through
print("\n.env included   :", ".env" in agmini.collect(ROOT))
print(".env if asked   :", ".env" in agmini.collect(ROOT, secrets=True))

everything      : 14
only .py        : 13
py+md, no tests : ['README.md', 'src/__future__.py', 'src/_aix_support.py', 'src/_collections_abc.py'] ...

.env included   : False
.env if asked   : True


## 2. `pack` — write the sheets

In [4]:
info = agmini.pack(ROOT, ROOT.parent / "sheets.html",
                   include=[".py", ".md"], exclude=["test_*.py"])
for k, v in info.items():
    print(f"  {k:<18} {v}")

  path               /tmp/agmini-3tjjhjcg/sheets.html
  files              13
  raw_bytes          137530
  compressed_bytes   34156
  frames             58
  sheets             8
  sha256             a53d74738f4215d613b193c321868dcfed5cb52a090561c655d47e9f3acccde0
  transfer_id        a53d7473


`sheets` is the number of photos you need to take. Open the HTML fullscreen and
shoot one screen at a time.

## 3. Tuning for your display

`scale` is screen pixels per QR module — the setting that decides whether a photo can
be decoded at all. Below about 3, the display physically cannot show the detail, and
no camera can recover it. The defaults suit 1080p; a bigger screen carries more per
sheet.

In [5]:
for label, kw in [("1080p default   ", {}),
                  ("1440p, bigger QR", dict(chunk=1100)),
                  ("4K, denser grid ", dict(chunk=1100, cols=6, rows=3))]:
    r = agmini.pack(ROOT, ROOT.parent / "t.html", include=[".py"], **kw)
    print(f"  {label}  {r['sheets']:>3} photos   ({r['frames']} frames)")

  1080p default       8 photos   (58 frames)


  1440p, bigger QR    5 photos   (33 frames)


  4K, denser grid     2 photos   (33 frames)


If you'd rather have this computed than guessed, `agpack.py` does it for a given
resolution:

```bash
python3 agpack.py --root ~/repo --ext .py,.md --screen 2560x1440 --estimate
```

## 4. Verify without a camera

The frames are text, so you can prove the payload survives encoding before spending
an hour photographing anything.

In [6]:
html = open(info["path"]).read()
rendered = html.count("<div class=c>")
print(f"symbols in the sheet file : {rendered}")
print(f"frames pack() reported    : {info['frames']}")
print("every frame drawn         :", rendered == info["frames"])

# same input -> same bytes, because the archive zeroes timestamps and ownership
again = agmini.pack(ROOT, ROOT.parent / "again.html",
                    include=[".py", ".md"], exclude=["test_*.py"])
print("\ndeterministic             :", again["sha256"] == info["sha256"])

symbols in the sheet file : 58
frames pack() reported    : 58
every frame drawn         : True



deterministic             : True


Determinism matters here: timestamps and ownership are zeroed in the archive, so the
same input always produces the same payload and the same transfer id. If you re-pack
after fixing one file, only that changes.

## Clean up

In [7]:
print(f"import shutil; shutil.rmtree('{ROOT.parent}')")

import shutil; shutil.rmtree('/tmp/agmini-3tjjhjcg')
